In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine

import pandas as pd, numpy as np
BASE  = '/content/drive/MyDrive/Palestine_Project'
RAW   = f'{BASE}/00_raw/Palestine_Project_Data.xlsx'
CLEAN = f'{BASE}/01_clean'
ML    = f'{BASE}/02_ml'

def read_sheet(name, **kw):
    return pd.read_excel(RAW, sheet_name=name, engine='calamine', **kw)

def save_clean(df, filename):
    df.to_csv(f'{CLEAN}/{filename}', index=False, encoding='utf-8')
    print(f'saved {filename}: {len(df):,} rows x {df.shape[1]} cols')

In [ ]:
g = read_sheet('6_Casualties_Gaza');     g['report_date'] = pd.to_datetime(g['report_date'])
w = read_sheet('6_Casualties_WestBank'); w['report_date'] = pd.to_datetime(w['report_date'])

for name, t in [('Gaza', g), ('West Bank', w)]:
    full = pd.date_range(t['report_date'].min(), t['report_date'].max())
    assert len(t) == len(full) == 1137 and t['report_date'].is_unique, name

G = pd.DataFrame({
    'date_key': g['report_date'].dt.strftime('%Y%m%d').astype(int), 'geo_pcode': 'PS02',
    'killed_cum': g['ext_killed_cum'],   'killed_daily': g['ext_killed'],
    'injured_cum': g['ext_injured_cum'], 'injured_daily': g['ext_injured'],
    'children_killed_cum': g['ext_killed_children_cum'], 'women_killed_cum': g['ext_killed_women_cum'],
    'medical_killed_cum': g['ext_med_killed_cum'],       'press_killed_cum': g['ext_press_killed_cum'],
    'civil_defense_killed_cum': g['ext_civdef_killed_cum'], 'massacres_cum': g['ext_massacres_cum'],
    'famine_deaths_cum': g['famine_cum'], 'child_famine_deaths_cum': g['child_famine_cum'],
    'aid_seekers_killed_cum': g['aid_seeker_killed_cum'], 'aid_seekers_injured_cum': g['aid_seeker_injured_cum'],
    'report_source': g['report_source'], 'report_period_hours': g['report_period'],
})
G['is_reported']         = (g['report_source'] != 'missing').astype(int)
G['is_filled']           = g['killed_cum'].isna().astype(int)
G['is_multi_day_report'] = (g['report_period'] >= 48).astype(int)
med = g['ext_killed'].rolling(15, center=True, min_periods=8).median()
G['is_backlog_spike']    = ((g['ext_killed'] >= 300) & (g['ext_killed'] > 5 * med)).astype(int)

assert (g['ext_killed_cum'].diff().fillna(g['ext_killed_cum']) == g['ext_killed']).all()
assert (g['ext_injured_cum'].diff().fillna(g['ext_injured_cum']) == g['ext_injured']).all()

W = pd.DataFrame({
    'date_key': w['report_date'].dt.strftime('%Y%m%d').astype(int), 'geo_pcode': 'PS01',
    'killed_cum': w['killed_cum'],   'killed_daily':  w['killed_cum'].diff().fillna(w['killed_cum']),
    'injured_cum': w['injured_cum'], 'injured_daily': w['injured_cum'].diff().fillna(w['injured_cum']),
    'children_killed_cum': w['killed_children_cum'],
    'settler_attacks_cum': w['settler_attacks_cum'], 'displaced_persons_cum': w['displaced_persons_cum'],
    'report_source': w['flash_source'],
})
W['is_reported'] = (w['flash_source'] == 'un').astype(int)
W['is_filled']   = (w['flash_source'] == 'fill').astype(int)

F = pd.concat([G, W], ignore_index=True)
F = F[['date_key','geo_pcode','killed_cum','killed_daily','injured_cum','injured_daily','children_killed_cum','women_killed_cum',
       'medical_killed_cum','press_killed_cum','civil_defense_killed_cum','massacres_cum','famine_deaths_cum',
       'child_famine_deaths_cum','aid_seekers_killed_cum','aid_seekers_injured_cum','settler_attacks_cum',
       'displaced_persons_cum','report_source','report_period_hours','is_reported','is_filled','is_multi_day_report',
       'is_backlog_spike']]
for c in ['killed_cum','killed_daily','injured_cum','injured_daily','children_killed_cum','women_killed_cum',
          'medical_killed_cum','press_killed_cum','civil_defense_killed_cum','massacres_cum','famine_deaths_cum',
          'child_famine_deaths_cum','aid_seekers_killed_cum','aid_seekers_injured_cum','settler_attacks_cum',
          'displaced_persons_cum','report_period_hours']:
    F[c] = F[c].astype('Int64')
for c in ['is_multi_day_report','is_backlog_spike']:
    F[c] = F[c].astype('Int64')

print('rows', len(F), '| Gaza last killed_cum', int(G['killed_cum'].iloc[-1]), '| Gaza last injured_cum', int(G['injured_cum'].iloc[-1]),
      '| WB last killed_cum', int(W['killed_cum'].iloc[-1]), '| WB last settler_attacks_cum', int(W['settler_attacks_cum'].iloc[-1]))
print('Gaza days not reported:', int((1 - G['is_reported']).sum()), '| filled:', int(G['is_filled'].sum()),
      '| multi-day reports:', int(G['is_multi_day_report'].sum()), '| backlog-spike days:', int(G['is_backlog_spike'].sum()))
print('Backlog-spike days:', g.loc[G['is_backlog_spike'] == 1, 'report_date'].dt.strftime('%Y-%m-%d').tolist())
print('WB rows copied forward (fill):', int(W['is_filled'].sum()), 'of', len(W))
for c in ['press_killed_cum']:
    print(' Gaza', c, 'decreases:', int((G[c].astype('float').diff() < 0).sum()))
print(' West Bank injured_children_cum decreases (not kept):', int((w['injured_children_cum'].diff() < 0).sum()))
save_clean(F, 'fact_casualties_daily.csv')